### Example for multiagent for creating form creation page.

In [21]:
import os
from typing import Annotated
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor

from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langchain_core.tools import Tool
from pprint import pprint

## File path
base_path = "./form/index.html"

def read_file(_unused=None) -> str:
    """Reads and returns the entire file content."""
    with open(base_path, "r", encoding="utf-8") as f:
        return f.read()

def write_file(content: str) -> str:
    """Writes content to a file (overwrites)."""
    with open(base_path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Written to {base_path}"

# Empty the file initially
write_file("")

# Tools
read_file_tool = Tool(
    name="read_the_html_file",
    description="Reads and returns the entire html file content",
    func=read_file
)
write_file_tool = Tool(
    name="write_the_html_file",
    description="Writes content to a file. note: will replace the entire file.",
    func=write_file
)

# Load API key
load_dotenv()
llm = ChatGroq(model_name="openai/gpt-oss-20b", api_key=os.getenv("GROQ_API_KEY"))

# Agents
planner_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are a Planner agent.
Your job: Plan the steps required to create an HTML page based on user input.
Do NOT write any HTML yourself. Only provide step-by-step plan.
""",
    name="planner_agent"
)

developer_agent = create_agent(
    model=llm,
    tools=[read_file_tool, write_file_tool],
    system_prompt="""
You are a UI developer agent.
Your job: Create a single HTML page based on the user's requirement.
You MUST NOT return code in messages.
You MUST use tools to read/write the HTML file.
""",
    name="developer_agent"
)

qa_agent = create_agent(
    model=llm,
    tools=[read_file_tool, write_file_tool],
    system_prompt="""
You are a QA agent.
Your job: Validate whether the HTML file satisfies user requirements.
You MUST NOT return code.
You MUST use tools to read the HTML file.
If issues found → instruct developer_agent.
""",
    name="qa_agent"
)

# Supervisor
supervisor = create_supervisor(
    agents=[planner_agent, developer_agent, qa_agent],
    model=llm,
    prompt="""
You are a Supervisor agent.
Rules:
1. Decide which agent should act next: planner_agent, developer_agent, or qa_agent.
2. Respond ONLY with the agent's name.
3. Respond with FINISH when the HTML page is complete and validated.
""",
).compile()

# State definition
class State(TypedDict):
    messages: Annotated[list, add_messages]
    planner_comment: str
    developer_comment: str
    qa_comment: str
    supervisor_decision: str

# Node wrappers
def planner_node(state: State):
    result = planner_agent.invoke(state)
    new_state = {
        **state,
        "messages": result["messages"],
        "planner_comment": result["messages"][-1].content  # fixed
    }
    print("\n--- Planner Node ---")
    print(f"Planner Response: {new_state['planner_comment']}")
    return new_state

def developer_node(state: State):
    result = developer_agent.invoke(state)
    new_state = {
        **state,
        "messages": result["messages"],
        "developer_comment": result["messages"][-1].content  # fixed
    }
    print("\n--- Developer Node ---")
    print(f"Developer Response: {new_state['developer_comment']}")
    return new_state

def qa_node(state: State):
    result = qa_agent.invoke(state)
    new_state = {
        **state,
        "messages": result["messages"],
        "qa_comment": result["messages"][-1].content  # fixed
    }
    print("\n--- QA Node ---")
    print(f"QA Response: {new_state['qa_comment']}")
    return new_state

def supervisor_node(state: State):
    result = supervisor.invoke(state)
    new_state = {
        **state,
        "messages": result["messages"],
        "supervisor_decision": result["messages"][-1].content  # fixed
    }
    print("\n--- Supervisor Node ---")
    print(f"Supervisor Decision: {new_state['supervisor_decision']}")
    return new_state

# Graph
graph_builder = StateGraph(State)

graph_builder.add_node("planner_agent", planner_agent)
graph_builder.add_node("developer_agent", developer_agent)
graph_builder.add_node("qa_agent", qa_agent)
graph_builder.add_node("supervisor", supervisor)

graph_builder.add_edge(START, "supervisor")

graph = graph_builder.compile()

# Initial state
user_prompt = "Create a beautiful form page to gather all personal information such as name, age, and email."

initial_state = {
    "messages": [{"role": "user", "content": user_prompt}],
    "planner_comment": "",
    "developer_comment": "",
    "qa_comment": "",
    "supervisor_decision": ""
}

# Invoke the graph
response = graph.invoke(initial_state)

print("\n=== Final Graph Response ===")
pprint(response)



=== Final Graph Response ===
{'developer_comment': '',
 'messages': [HumanMessage(content='Create a beautiful form page to gather all personal information such as name, age, and email.', additional_kwargs={}, response_metadata={}, id='3078d8d8-b9ce-4cf9-abcb-7c2098abe0b8'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user requests: "Create a beautiful form page to gather all personal information such as name, age, and email." The instruction says: I am a Supervisor agent. I must decide which agent should act next: planner_agent, developer_agent, or qa_agent. I must respond only with the agent\'s name. So I should pick the appropriate agent. The user wants to create a form page. This is a development task: we need to write code. So we should transfer to developer_agent.', 'tool_calls': [{'id': 'fc_9a3041c4-acb1-4779-9fc0-7a51f87768ad', 'function': {'arguments': '{}', 'name': 'transfer_to_developer_agent'}, 'type': 'function'}]}, response_metadata={'